# Data Profiling and Sanity Check Analysis
## Transportation & Logistics Dataset


Importing libraries and dataset

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('Dataset/raw_data.csv')

## **1. DATASET STRUCTURE ANALYSIS**

In [3]:
print("=" * 80)
print("DATASET STRUCTURE ANALYSIS")
print("=" * 80)
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nColumn names and data types:")
print(df.dtypes)

DATASET STRUCTURE ANALYSIS
Dataset shape: (69975, 12)
Number of rows: 69,975
Number of columns: 12

Column names and data types:
order_id                object
origin_city             object
destination_city        object
distance_km            float64
vehicle_type            object
order_date              object
order_hour               int64
weekday                 object
weather                 object
temperature            float64
traffic_level           object
delivery_time_hours    float64
dtype: object


## **2. DATA QUALITY ANALYSIS**


In [4]:
print("\n" + "=" * 80)
print("DATA QUALITY ANALYSIS")
print("=" * 80)

# Missing values
print("\nMissing values:")
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(4)
})
print(missing_data[missing_data['Missing_Count'] > 0])

# Duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Duplicate order_ids: {df['order_id'].duplicated().sum()}")



DATA QUALITY ANALYSIS

Missing values:
Empty DataFrame
Columns: [Column, Missing_Count, Missing_Percentage]
Index: []

Duplicate rows: 0
Duplicate order_ids: 52410


## **3. DESCRIPTIVE STATISTICS**


In [5]:
print("\n" + "=" * 80)
print("DESCRIPTIVE STATISTICS")
print("=" * 80)

# Numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumerical columns: {numerical_cols}")
print(df[numerical_cols].describe())

# Categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns: {categorical_cols}")
for col in categorical_cols:
    print(f"\n{col} - Unique values: {df[col].nunique()}")
    print(df[col].value_counts().head())


DESCRIPTIVE STATISTICS

Numerical columns: ['distance_km', 'order_hour', 'temperature', 'delivery_time_hours']
        distance_km    order_hour   temperature  delivery_time_hours
count  69975.000000  69975.000000  69975.000000         69975.000000
mean     575.202222     11.465666      3.633442             9.153710
std      366.703684      6.910821      3.715461             6.035765
min       69.700000      0.000000     -4.000000             0.830000
25%      200.000000      6.000000      4.000000             3.780000
50%      527.400000     11.000000      4.700000             8.050000
75%      808.400000     17.000000      5.900000            13.370000
max     1312.300000     23.000000      7.200000            29.840000

Categorical columns: ['order_id', 'origin_city', 'destination_city', 'vehicle_type', 'order_date', 'weekday', 'weather', 'traffic_level']

order_id - Unique values: 17565
order_id
ORD-20260105133858-3094    15
ORD-20260105133858-4285    14
ORD-20260105133858-6122   

## **4. TARGET VARIABLE ANALYSIS**


In [6]:
target_col = 'delivery_time_hours'

print("\n" + "=" * 80)
print("TARGET VARIABLE ANALYSIS")
print("=" * 80)

print(f"\nTarget variable: {target_col}")
print(df[target_col].describe())

# Outlier detection
Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[target_col] < lower_bound) | (df[target_col] > upper_bound)]
print(f"\nOutliers (IQR method): {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
print(f"Outlier range: {outliers[target_col].min():.2f} to {outliers[target_col].max():.2f}")

# Distribution characteristics
print(f"\nDistribution characteristics:")
print(f"Skewness: {df[target_col].skew():.4f}")
print(f"Kurtosis: {df[target_col].kurtosis():.4f}")


TARGET VARIABLE ANALYSIS

Target variable: delivery_time_hours
count    69975.000000
mean         9.153710
std          6.035765
min          0.830000
25%          3.780000
50%          8.050000
75%         13.370000
max         29.840000
Name: delivery_time_hours, dtype: float64

Outliers (IQR method): 171 (0.24%)
Outlier range: 27.76 to 29.84

Distribution characteristics:
Skewness: 0.6551
Kurtosis: -0.3489


## **5. RED FLAGS DETECTION**


In [7]:
print("\n" + "=" * 80)
print("RED FLAGS DETECTION")
print("=" * 80)

red_flags = []

# Impossible values
if (df['delivery_time_hours'] < 0).sum() > 0:
    red_flags.append("Negative delivery times detected")
if (df['distance_km'] < 0).sum() > 0:
    red_flags.append("Negative distances detected")

# Duplicate order IDs
duplicate_rate = df['order_id'].duplicated().sum() / len(df) * 100
if duplicate_rate > 10:
    red_flags.append(f"High duplicate order ID rate: {duplicate_rate:.1f}%")

# Same origin-destination
same_city = (df['origin_city'] == df['destination_city']).sum()
if same_city > 0:
    red_flags.append(f"Same origin-destination orders: {same_city}")

print("Red flags identified:")
for i, flag in enumerate(red_flags, 1):
    print(f"{i}. {flag}")

print("\nAnalysis complete.")


RED FLAGS DETECTION
Red flags identified:
1. High duplicate order ID rate: 74.9%

Analysis complete.
